# 面试问题：EGNN 怎样只用距离和相对坐标实现 E(n) 等变的分子坐标更新？

## 可直接复述的回答主线

1. 普通坐标 MLP 直接读取绝对 x/y，平移或旋转后输出不一定同步变换。
2. EGNN 的边消息使用节点特征和平方距离，坐标更新只沿相对向量 x_i-x_j 乘标量系数。
3. 距离在旋转和平移下不变，相对向量随旋转等变，因此聚合坐标更新天然满足 E(n) 等变。
4. 消息传递要显式构造双向边、按 receiver 聚合并按度归一化，避免高连接节点更新过大。
5. 评测既要看去噪 RMSE，也要把同一分子旋转/平移后再次 forward，直接测 equivariance error。
6. 生产还需三维原子类型、周期边界、邻居截断、力/能量一致性、长程相互作用和物理验证集。

后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例使用六碳环骨架的 9 个构象：6 个训练构象和 3 个测试构象。每个构象有 6 个碳节点、6 条环键，输入坐标经过旋转、平移和径向缩放，目标是恢复未缩放坐标。二维仅为教学可视化，不能代表真实分子动力学。

In [1]:
import math  # 生成六边形坐标并计算 RMSE。
import torch  # 使用基础张量和自动微分实现 EGNN 消息传递。
torch.manual_seed(103)  # 固定网络初始化和训练轨迹。
atom_names = ["C1", "C2", "C3", "C4", "C5", "C6"]  # 定义六个碳原子节点。
undirected_edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0)]  # 定义六碳环的六条化学键。
directed_edges = [(source, target) for source, target in undirected_edges for source, target in ((source, target), (target, source))]  # 把每条键展开为双向消息边。
base_coordinates = torch.tensor([[math.cos(index * math.pi / 3.0), math.sin(index * math.pi / 3.0)] for index in range(6)], dtype=torch.float32)  # 构造单位六边形参考坐标。
node_features = torch.tensor([[0.6, 1.0] for _ in atom_names], dtype=torch.float32)  # 用原子序数归一值和环内度构造两维特征。
conformation_specs = [("conf-01", 0.00, 0.82, [0.0, 0.0], "train"), ("conf-02", 0.30, 1.18, [0.1, -0.1], "train"), ("conf-03", 0.60, 0.90, [-0.1, 0.1], "train"), ("conf-04", 0.90, 1.12, [0.2, 0.0], "train"), ("conf-05", 1.20, 0.86, [0.0, -0.2], "train"), ("conf-06", 1.50, 1.20, [-0.2, 0.0], "train"), ("conf-07", 2.00, 0.88, [3.0, -2.0], "test"), ("conf-08", 2.50, 1.14, [-2.0, 2.5], "test"), ("conf-09", 3.00, 0.78, [4.0, 1.5], "test")]  # 定义六个训练与三个大平移测试构象。
conformations = []  # 保存输入、目标和变换参数。
for conformation_id, angle, scale, translation_values, split in conformation_specs:  # 逐规格生成旋转和平移构象。
    rotation = torch.tensor([[math.cos(angle), -math.sin(angle)], [math.sin(angle), math.cos(angle)]], dtype=torch.float32)  # 构造二维旋转矩阵。
    translation = torch.tensor(translation_values, dtype=torch.float32)  # 构造平移向量。
    target = base_coordinates @ rotation.T + translation  # 生成保持键长的目标构象。
    noisy = scale * (base_coordinates @ rotation.T) + translation  # 只缩放相对结构并保留全局平移。
    conformations.append({"id": conformation_id, "angle": angle, "scale": scale, "translation": translation, "split": split, "input": noisy, "target": target})  # 保存当前坐标去噪样本。
training_conformations = [item for item in conformations if item["split"] == "train"]  # 取得六个训练构象。
test_conformations = [item for item in conformations if item["split"] == "test"]  # 取得三个旋转且大平移测试构象。
print("教学实验输入：六碳环坐标去噪")  # 标记下方为受控分子图数据。
print("构象      split  angle  scale  translation  C1 input -> target")  # 输出构象预览表头。
for item in conformations:  # 逐构象展示变换和第一个原子坐标。
    print(f"{item['id']:<9} {item['split']:<5} {item['angle']:>6.2f} {item['scale']:>6.2f} {item['translation'].tolist()!s:<14} {item['input'][0].tolist()} -> {item['target'][0].tolist()}")  # 输出当前构象的真实坐标任务。
print("directed_edges=", directed_edges)  # 展示手写消息传递使用的十二条有向边。

教学实验输入：六碳环坐标去噪
构象      split  angle  scale  translation  C1 input -> target
conf-01   train   0.00   0.82 [0.0, 0.0]     [0.8199999928474426, 0.0] -> [1.0, 0.0]
conf-02   train   0.30   1.18 [0.10000000149011612, -0.10000000149011612] [1.227297067642212, 0.24871385097503662] -> [1.0553364753723145, 0.19552022218704224]
conf-03   train   0.60   0.90 [-0.10000000149011612, 0.10000000149011612] [0.6428020000457764, 0.6081782579421997] -> [0.7253355979919434, 0.6646425127983093]
conf-04   train   0.90   1.12 [0.20000000298023224, 0.0] [0.8962031602859497, 0.8773261308670044] -> [0.8216099739074707, 0.7833269238471985]
conf-05   train   1.20   0.86 [0.0, -0.20000000298023224] [0.31162768602371216, 0.6015536189079285] -> [0.3623577654361725, 0.7320390939712524]
conf-06   train   1.50   1.20 [-0.20000000298023224, 0.0] [-0.11511535942554474, 1.1969940662384033] -> [-0.1292628049850464, 0.9974949955940247]
conf-07   test    2.00   0.88 [3.0, -2.0]    [2.6337907314300537, -1.1998183727264404] -

## 2. Baseline / 基线：绝对坐标 MLP 预测每个原子位移

Baseline 把节点特征和绝对 x/y 拼接后预测 delta。它在小平移训练集上可拟合，但测试构象整体移动数个单位后会把平移误当作结构形变。

In [2]:
class AbsoluteCoordinateMLP(torch.nn.Module):  # 定义显式 forward 的非等变坐标基线。
    def __init__(self):  # 初始化逐节点 MLP。
        super().__init__()  # 注册 PyTorch 参数管理。
        self.network = torch.nn.Sequential(torch.nn.Linear(4, 24), torch.nn.SiLU(), torch.nn.Linear(24, 2))  # 从节点特征加绝对坐标预测二维位移。
    def forward(self, features, coordinates):  # 对六个原子独立执行绝对坐标回归。
        inputs = torch.cat([features, coordinates], dim=1)  # 拼接原子类型与绝对位置。
        displacement = self.network(inputs)  # 预测每个原子坐标修正量。
        return coordinates + displacement  # 返回去噪坐标。
baseline_model = AbsoluteCoordinateMLP()  # 创建非等变基线模型。
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.018)  # 创建小型 Adam 更新器。
baseline_history = []  # 保存真实 backward 训练轨迹。
for step in range(320):  # 在六个训练构象上执行全批次训练。
    baseline_optimizer.zero_grad(set_to_none=True)  # 清除上一步梯度。
    predictions = [baseline_model(node_features, item["input"]) for item in training_conformations]  # 前向预测六个构象坐标。
    loss = torch.stack([((prediction - item["target"]) ** 2).mean() for prediction, item in zip(predictions, training_conformations)]).mean()  # 计算平均坐标 MSE。
    loss.backward()  # 对绝对坐标 MLP 执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in baseline_model.parameters()))  # 汇总参数梯度范数。
    baseline_optimizer.step()  # 应用 Adam 参数更新。
    if step % 80 == 0 or step == 319:  # 每八十步保存训练轨迹。
        baseline_history.append({"step": step, "loss": loss.item(), "gradient_norm": gradient_norm})  # 保存损失和梯度。
with torch.no_grad():  # 在测试集评估绝对坐标模型。
    baseline_test_predictions = [baseline_model(node_features, item["input"]) for item in test_conformations]  # 计算三个大平移构象预测。
baseline_test_rmse = math.sqrt(sum(float(((prediction - item["target"]) ** 2).sum().item()) for prediction, item in zip(baseline_test_predictions, test_conformations)) / (len(test_conformations) * len(atom_names) * 2))  # 计算所有测试坐标 RMSE。
print("Baseline训练轨迹=", baseline_history)  # 展示真实 forward/backward 收敛。
print(f"Baseline大平移测试RMSE={baseline_test_rmse:.6f}")  # 展示绝对坐标泛化失败。

Baseline训练轨迹= [{'step': 0, 'loss': 0.03179469332098961, 'gradient_norm': 0.26844499688704326}, {'step': 80, 'loss': 0.011692151427268982, 'gradient_norm': 0.0007632933300947667}, {'step': 160, 'loss': 0.011453903280198574, 'gradient_norm': 0.0007142570546919055}, {'step': 240, 'loss': 0.007584432605654001, 'gradient_norm': 0.029882513267602127}, {'step': 319, 'loss': 0.004318197723478079, 'gradient_norm': 0.0024222318179346314}]
Baseline大平移测试RMSE=1.826032


## 3. 底层实现：距离消息与相对向量坐标更新

边 MLP 只看 h_i、h_j、||x_i-x_j||²；坐标 MLP 输出标量，乘相对向量后按 receiver 度归一化。下面堆叠两层并真实训练。

In [3]:
class EGNNLayer(torch.nn.Module):  # 定义一层 E(n) 等变消息传递。
    def __init__(self, hidden_dim):  # 初始化边、坐标和节点更新网络。
        super().__init__()  # 注册所有子模块参数。
        self.edge_mlp = torch.nn.Sequential(torch.nn.Linear(hidden_dim * 2 + 1, hidden_dim), torch.nn.SiLU(), torch.nn.Linear(hidden_dim, hidden_dim), torch.nn.SiLU())  # 从节点对和距离生成边消息。
        self.coordinate_mlp = torch.nn.Sequential(torch.nn.Linear(hidden_dim, hidden_dim), torch.nn.SiLU(), torch.nn.Linear(hidden_dim, 1), torch.nn.Tanh())  # 从不变边消息生成有界标量系数。
        self.node_mlp = torch.nn.Sequential(torch.nn.Linear(hidden_dim * 2, hidden_dim), torch.nn.SiLU(), torch.nn.Linear(hidden_dim, hidden_dim))  # 用聚合边消息更新节点特征。
    def forward(self, hidden, coordinates, edges, return_debug=False):  # 对一个分子图执行显式消息聚合。
        receivers = torch.tensor([edge[0] for edge in edges], dtype=torch.long)  # 构造每条有向边的接收节点索引。
        neighbors = torch.tensor([edge[1] for edge in edges], dtype=torch.long)  # 构造每条有向边的邻居索引。
        relative = coordinates[receivers] - coordinates[neighbors]  # 计算随旋转等变的相对向量。
        squared_distance = (relative ** 2).sum(dim=1, keepdim=True)  # 计算旋转和平移不变的平方距离。
        edge_inputs = torch.cat([hidden[receivers], hidden[neighbors], squared_distance], dim=1)  # 拼接两端特征和不变距离。
        messages = self.edge_mlp(edge_inputs)  # 生成每条边的不变消息。
        coefficients = self.coordinate_mlp(messages) * 0.25  # 生成受限坐标更新标量以稳定训练。
        coordinate_messages = relative * coefficients  # 用标量缩放相对向量保持等变性。
        coordinate_sum = torch.zeros_like(coordinates)  # 初始化每个 receiver 的坐标消息和。
        coordinate_sum.index_add_(0, receivers, coordinate_messages)  # 按 receiver 聚合相对坐标更新。
        degrees = torch.zeros(coordinates.shape[0], 1, dtype=coordinates.dtype)  # 初始化有向入度计数。
        degrees.index_add_(0, receivers, torch.ones(len(edges), 1, dtype=coordinates.dtype))  # 累加每个 receiver 的消息条数。
        new_coordinates = coordinates + coordinate_sum / degrees.clamp_min(1.0)  # 用度归一化坐标更新避免高连接放大。
        message_sum = torch.zeros_like(hidden)  # 初始化节点边消息聚合。
        message_sum.index_add_(0, receivers, messages)  # 按 receiver 汇总不变边消息。
        message_mean = message_sum / degrees.clamp_min(1.0)  # 按度求消息平均。
        new_hidden = hidden + self.node_mlp(torch.cat([hidden, message_mean], dim=1))  # 用残差节点 MLP 更新隐藏特征。
        if return_debug:  # 检查调用方是否需要边级中间量。
            return new_hidden, new_coordinates, {"relative": relative, "squared_distance": squared_distance, "messages": messages, "coefficients": coefficients, "degrees": degrees}  # 返回边距离、消息和系数。
        return new_hidden, new_coordinates  # 返回更新后的节点特征与坐标。
class EGNNDenoiser(torch.nn.Module):  # 定义两层显式 forward 的坐标去噪器。
    def __init__(self, input_dim=2, hidden_dim=24):  # 初始化节点编码和两层 EGNN。
        super().__init__()  # 注册 PyTorch 模块参数。
        self.input_projection = torch.nn.Linear(input_dim, hidden_dim)  # 把原子特征映射到隐藏空间。
        self.layer_one = EGNNLayer(hidden_dim)  # 创建第一层等变消息传递。
        self.layer_two = EGNNLayer(hidden_dim)  # 创建第二层等变消息传递。
    def forward(self, features, coordinates, edges, return_debug=False):  # 对一幅分子构象执行两层坐标更新。
        hidden = torch.relu(self.input_projection(features))  # 初始化节点隐藏特征。
        hidden, updated_coordinates, first_debug = self.layer_one(hidden, coordinates, edges, return_debug=True)  # 执行第一层并保留边中间量。
        hidden, updated_coordinates = self.layer_two(hidden, updated_coordinates, edges)  # 执行第二层等变更新。
        if return_debug:  # 检查是否需要第一层解释信息。
            return updated_coordinates, first_debug  # 返回最终坐标和边消息证据。
        return updated_coordinates  # 返回去噪坐标。
egnn_model = EGNNDenoiser()  # 创建两层 EGNN 去噪模型。
egnn_optimizer = torch.optim.Adam(egnn_model.parameters(), lr=0.012)  # 创建参数更新器。
egnn_history = []  # 保存实际 backward 轨迹。
for step in range(420):  # 在六个构象上执行全批次等变训练。
    egnn_optimizer.zero_grad(set_to_none=True)  # 清除上一步梯度。
    predictions = [egnn_model(node_features, item["input"], directed_edges) for item in training_conformations]  # 前向预测六个训练坐标。
    loss = torch.stack([((prediction - item["target"]) ** 2).mean() for prediction, item in zip(predictions, training_conformations)]).mean()  # 计算坐标均方误差。
    loss.backward()  # 对边、坐标和节点网络执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in egnn_model.parameters() if parameter.grad is not None))  # 汇总所有参与坐标损失的非空参数梯度范数。
    egnn_optimizer.step()  # 应用 Adam 参数更新。
    if step % 105 == 0 or step == 419:  # 每一百零五步保存训练轨迹。
        egnn_history.append({"step": step, "loss": loss.item(), "gradient_norm": gradient_norm})  # 保存损失和梯度。
with torch.no_grad():  # 在测试构象上执行最终推理。
    egnn_test_predictions = [egnn_model(node_features, item["input"], directed_edges) for item in test_conformations]  # 预测三个大平移构象。
    preview_prediction, preview_debug = egnn_model(node_features, training_conformations[0]["input"], directed_edges, return_debug=True)  # 取得第一构象边级中间量。
egnn_test_rmse = math.sqrt(sum(float(((prediction - item["target"]) ** 2).sum().item()) for prediction, item in zip(egnn_test_predictions, test_conformations)) / (len(test_conformations) * len(atom_names) * 2))  # 计算 EGNN 测试坐标 RMSE。
print("EGNN训练轨迹=", egnn_history)  # 展示损失和梯度变化。
print("第一层前四条边 distance²=", torch.round(preview_debug["squared_distance"][:4, 0] * 10000).tolist())  # 展示距离不变量的数值。
print("第一层前四条边 coordinate coefficients=", torch.round(preview_debug["coefficients"][:4, 0] * 10000).tolist())  # 展示相对坐标缩放系数。

EGNN训练轨迹= [{'step': 0, 'loss': 0.012591526843607426, 'gradient_norm': 0.008424763800807174}, {'step': 105, 'loss': 7.0136011345312e-06, 'gradient_norm': 0.0014271535101382854}, {'step': 210, 'loss': 6.647774625889724e-06, 'gradient_norm': 0.0034831802376723}, {'step': 315, 'loss': 4.404491392051568e-06, 'gradient_norm': 0.002269097066808753}, {'step': 419, 'loss': 1.2101136235287413e-05, 'gradient_norm': 0.003878226795628967}]
第一层前四条边 distance²= [6724.0, 6724.0, 6724.0, 6724.0]
第一层前四条边 coordinate coefficients= [2339.0, 2339.0, 2339.0, 2339.0]


## 4. 逐构象结果与结果解读

对三个未见旋转且大平移构象比较绝对坐标 MLP 与 EGNN 的坐标 RMSE，并展示 C1 预测。

In [4]:
print("构象      translation      baseline_RMSE  EGNN_RMSE  C1 target / baseline / EGNN")  # 输出逐构象对照表头。
for item, baseline_prediction, egnn_prediction in zip(test_conformations, baseline_test_predictions, egnn_test_predictions):  # 逐测试构象比较两种模型。
    baseline_rmse = float(torch.sqrt(((baseline_prediction - item["target"]) ** 2).mean()).item())  # 计算当前构象 Baseline RMSE。
    egnn_rmse = float(torch.sqrt(((egnn_prediction - item["target"]) ** 2).mean()).item())  # 计算当前构象 EGNN RMSE。
    print(f"{item['id']:<9} {item['translation'].tolist()!s:<16} {baseline_rmse:>13.5f} {egnn_rmse:>10.5f} {item['target'][0].tolist()} / {baseline_prediction[0].tolist()} / {egnn_prediction[0].tolist()}")  # 输出当前构象误差和实际坐标。
print(f"结果解读：绝对坐标MLP测试RMSE={baseline_test_rmse:.5f}，EGNN={egnn_test_rmse:.5f}；相对向量更新能跨大平移泛化。")  # 解释 E(n) 等变结构收益。

构象      translation      baseline_RMSE  EGNN_RMSE  C1 target / baseline / EGNN
conf-07   [3.0, -2.0]            2.03045    0.00306 [2.583853244781494, -1.0907025337219238] / [1.7613626718521118, 0.25443780422210693] / [2.582049608230591, -1.0867618322372437]
conf-08   [-2.0, 2.5]            1.48327    0.00141 [-2.8011436462402344, 3.0984721183776855] / [-0.8408539295196533, 1.418850064277649] / [-2.7995433807373047, 3.0972766876220703]
conf-09   [4.0, 1.5]             1.91842    0.02292 [3.010007381439209, 1.6411199569702148] / [1.297622561454773, 0.7315061092376709] / [3.042099952697754, 1.636545181274414]
结果解读：绝对坐标MLP测试RMSE=1.82603，EGNN=0.01338；相对向量更新能跨大平移泛化。


## 5. 失败案例与修正：整体平移后绝对坐标模型输出不平移

对同一测试输入再加 `[10,-7]`。理想输出应在原预测基础上加同一向量；直接测两种模型的平移等变误差。

In [5]:
probe = test_conformations[0]  # 选择 conf-07 作为等变性探针。
extra_translation = torch.tensor([10.0, -7.0], dtype=torch.float32)  # 定义从未见过的大额整体平移。
with torch.no_grad():  # 在无梯度环境执行两次对照 forward。
    baseline_original = baseline_model(node_features, probe["input"])  # 计算绝对坐标模型原输出。
    baseline_translated = baseline_model(node_features, probe["input"] + extra_translation)  # 计算整体平移后的输出。
    egnn_original = egnn_model(node_features, probe["input"], directed_edges)  # 计算 EGNN 原输出。
    egnn_translated = egnn_model(node_features, probe["input"] + extra_translation, directed_edges)  # 计算 EGNN 平移输入输出。
baseline_equivariance_error = float(torch.max(torch.abs(baseline_translated - (baseline_original + extra_translation))).item())  # 计算 Baseline 与理想平移输出的最大差。
egnn_equivariance_error = float(torch.max(torch.abs(egnn_translated - (egnn_original + extra_translation))).item())  # 计算 EGNN 平移等变最大差。
print(f"错误行为：Absolute MLP translation equivariance error={baseline_equivariance_error:.6f}")  # 展示绝对坐标依赖导致变换不一致。
print(f"修正行为：EGNN translation equivariance error={egnn_equivariance_error:.8f}")  # 展示相对坐标消息保持平移等变。

错误行为：Absolute MLP translation equivariance error=12.742109
修正行为：EGNN translation equivariance error=0.00000095


## 6. 生产边界

二维六碳环不含真实原子势能。生产需要三维元素/电荷/键类型、邻居截断与 cell list、周期边界、反射是否允许、能量与力梯度一致性、长程库仑作用、混合精度稳定性，以及按分子骨架而非随机构象切分验证集。

In [6]:
egnn_diagnostics = {"atoms": len(atom_names), "bonds": len(undirected_edges), "training_conformations": len(training_conformations), "test_conformations": len(test_conformations), "baseline_test_rmse": baseline_test_rmse, "egnn_test_rmse": egnn_test_rmse, "baseline_equivariance_error": baseline_equivariance_error, "egnn_equivariance_error": egnn_equivariance_error}  # 汇总图规模、去噪和等变性指标。
print("生产监控快照：", egnn_diagnostics)  # 输出 EGNN 管线应持续观察的信号。

生产监控快照： {'atoms': 6, 'bonds': 6, 'training_conformations': 6, 'test_conformations': 3, 'baseline_test_rmse': 1.8260319856226381, 'egnn_test_rmse': 0.013376825787189718, 'baseline_equivariance_error': 12.742109298706055, 'egnn_equivariance_error': 9.5367431640625e-07}


## 7. 最小回归测试

断言覆盖分子规模、真实训练、坐标去噪、等变性和有限输出。

In [7]:
assert len(atom_names) >= 6 and len(undirected_edges) >= 6 and len(conformations) >= 6  # 保证案例具有非平凡分子图和多个构象。
assert baseline_history[-1]["loss"] < baseline_history[0]["loss"] and egnn_history[-1]["loss"] < egnn_history[0]["loss"]  # 保证两种模型都实际训练收敛。
assert all(row["gradient_norm"] > 0.0 for row in egnn_history)  # 保证 EGNN 执行真实 backward 并产生梯度。
assert egnn_test_rmse < baseline_test_rmse  # 保证相同测试构象上 EGNN 优于绝对坐标基线。
assert egnn_equivariance_error < 1.0e-5 and baseline_equivariance_error > egnn_equivariance_error + 1.0e-3  # 保证平移失败真实复现且 EGNN 修正。
assert all(torch.isfinite(prediction).all() for prediction in egnn_test_predictions)  # 保证全部坐标输出有限。